# Exercise 03 — Capital allocation to risky assets

MSc Finance · Investments · FHNW · Autumn 2026

How much of your wealth belongs in the risky asset at all? This week the inputs are not
history but a forecast: the capital market assumptions a large asset manager publishes
every autumn and that pension funds across Europe feed into their strategic asset
allocation.

**How to work with this notebook.** The task text is here and on the exercise sheet. Each
code cell is a stub: the `# TODO` lines are the steps, in order. The setup and data cells
below are complete — run them first and leave them alone.

Run **Runtime → Restart and run all** before you trust any number in here.

**Data.** J.P. Morgan Asset Management, *2026 Long-Term Capital Market Assumptions*, 30th
annual edition, assumption matrix in **Swiss francs**, as of 30 September 2025. The file
holds Swiss cash and the complete equity block of the matrix. Source:
[am.jpmorgan.com/ch/en/asset-management/adv/insights/portfolio-insights/long-term-capital-market-assumptions](https://am.jpmorgan.com/ch/en/asset-management/adv/insights/portfolio-insights/long-term-capital-market-assumptions/).

Two columns of the matrix are returns, and the difference matters. The **arithmetic**
return is the input mean-variance analysis asks for; the **compound** return is what one
euro actually grows at, lower by roughly $\sigma^2/2$ — the gap you measured on historical
data in Exercise 02. Everything below uses the arithmetic column.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from exercise_utils import FHNW, setup_style, save_results
setup_style()

pct = "{:.2%}".format

In [ ]:
BASE = "https://raw.githubusercontent.com/KroeTiA/Investments/main/"
DATA_URL = BASE + "Exercise_03/data/ltcma_2026_chf_equities.csv"

# Four lines of provenance sit on top of the file. All figures are in percent per annum.
cma = pd.read_csv(DATA_URL, skiprows=4).set_index("Asset class")
cma = cma.rename(columns={"Arithmetic return 2026 (%)": "ER",
                          "Annualized volatility (%)": "SD",
                          "Compound return 2026 (%)": "Compound"})
cma[["ER", "SD", "Compound"]] /= 100.0

RF = cma.loc["Swiss Cash", "ER"]                    # Swiss cash — the risk-free asset
menu = cma.loc[cma["Category"] == "Equity", ["ER", "SD"]].copy()

P = "AC World Equity"                               # the pre-specified risky portfolio
ER_P, SD_P = menu.loc[P, "ER"], menu.loc[P, "SD"]

print(f"risk-free rate {RF:.2%}          {len(menu)} equity asset classes")
print(f"{P}:  E(R) {ER_P:.2%}   sigma {SD_P:.2%}   risk premium {ER_P - RF:.2%}")

## Task 1 — Scoring the menu

Two clients walk in. Roger has risk aversion $A = 2$, Martina has $A = 6$. Neither of them
holds cash: whatever they choose, they are fully invested in one equity asset class.

Score every asset class on the menu with the mean-variance utility function

$$U = E(R) - \tfrac{1}{2} A \sigma^2$$

and rank the seventeen asset classes for each client. Report the top three and the bottom
three for both.

Then find the risk aversion at which the two clients would agree. Take **Swiss Equity** and
**AC World Equity** — the home market and the global one — and solve for the $A$ at which
they are equally attractive. State which of the two an investor prefers above that value
and which below.

Finally, draw the menu in $(\sigma, E(R))$ space and add the indifference curve running
through the winning asset for each client. Choose the axis ranges yourself.

*Deliverable: the ranking table for both clients, the break-even $A$ with one sentence on
which side is which, and the figure with two indifference curves.*

In [ ]:
# The utility score of an asset with mean er and volatility sd.
def utility(er, sd, A):
    ...
    # TODO: one line — the utility function from the lecture
scores = menu.copy()
# TODO: a utility column and a rank column for each of the two clients
print(scores.sort_values("Rank (A=2)")
            .to_string(formatters={c: pct for c in scores.columns
                                   if not c.startswith("Rank")}))

In [ ]:
home, world = "Swiss Equity", "AC World Equity"
# TODO: the A at which the two assets deliver the same utility score

In [ ]:
grid = np.linspace(0.0, 0.32, 300)
LABEL = ["Swiss Equity", "AC World Equity", "Japanese Equity",
         "Chinese Domestic Equity", "Euro Area Small Cap", "U.S. Small Cap"]
fig1, ax = plt.subplots(figsize=(8.4, 5.0))
ax.scatter(menu["SD"], menu["ER"], s=28, color=FHNW["navy"], zorder=3)
for name in LABEL:
    ax.annotate(name, (menu.loc[name, "SD"], menu.loc[name, "ER"]),
                fontsize=8, xytext=(6, -4), textcoords="offset points")
for A, colour in [(2, FHNW["blue"]), (6, FHNW["orange"])]:
    ...
    # TODO: the indifference curve through the winner, E(R) = U + 0.5 * A * sigma^2
ax.set_xlabel("Volatility (annual)")
ax.set_ylabel("Expected return (annual, arithmetic)")
ax.set_xlim(0.10, 0.32)
ax.set_ylim(0.02, 0.13)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(loc="upper left")
plt.show()

## Task 2 — Adding cash changes the question

Now let both clients hold Swiss cash alongside the risky asset. A share $y$ goes into the
risky asset and $1 - y$ into cash, so the complete portfolio has

$$E(R_C) = R_f + y\,[E(R_P) - R_f], \qquad \sigma_C = y\,\sigma_P .$$

Write a function `utility_of_y(y, er, sd, A, rf)` that returns the utility score of the
complete portfolio. Evaluate it over a grid of $y$ from 0 to 2 for the pre-specified risky
portfolio $P$ = AC World Equity, once for each client, and plot the two curves. Mark the
maximum you read off the grid and check it against the closed form

$$y^* = \frac{E(R_P) - R_f}{A\,\sigma_P^2}.$$

Then go back to the full menu. For every asset class compute the Sharpe ratio, the optimal
share $y^*$ and the utility at the optimum $U^* = U(y^*)$, for both clients, and rank the
menu by $U^*$.

Compare that ranking with the two rankings from Task 1. **Say what changed, and explain
why.** One number in your table explains the whole thing.

*Deliverable: the $U(y)$ figure with four marked points, the menu table with $SR$, $y^*$ and
$U^*$ for both clients, and two or three sentences on why the ranking behaves as it does.*

In [ ]:
# Utility of a complete portfolio holding a share y of the risky asset.
def utility_of_y(y, er, sd, A, rf):
    ...
    # TODO: the mean and the volatility of the complete portfolio, then reuse utility()
ys = np.linspace(0, 2, 401)
fig2, ax = plt.subplots(figsize=(8.4, 4.6))
for A, colour in [(2, FHNW["blue"]), (6, FHNW["orange"])]:
    ...
    # TODO: the utility curve, the grid maximum, and the closed-form y*
    ax.plot(ys, u, color=colour, lw=1.8, label=f"A = {A}")
    ax.scatter([y_grid], [u.max()], s=45, color=colour, zorder=3)
ax.axhline(RF, color="#999999", lw=1.0, ls="--")
ax.set_xlabel("Share y in AC World Equity")
ax.set_ylabel("Utility score of the complete portfolio")
ax.legend()
plt.show()

In [ ]:
opt = menu.copy()
# TODO: the Sharpe ratio of each asset class
# TODO: y* and U* for each client, then rank the menu by U*
print(opt.sort_values("Rank (A=2)").to_string(
    formatters={"ER": pct, "SD": pct, "SR": "{:.3f}".format,
                "y* (A=2)": "{:.2f}".format, "y* (A=6)": "{:.2f}".format,
                "U* (A=2)": pct, "U* (A=6)": pct}))
# TODO: check the two U* rankings against each other and against the SR ranking

## Task 3 — How much does it cost to be wrong?

Capital market assumptions are forecasts, and J.P. Morgan revises them every autumn. Stay
with $P$ = AC World Equity and take a client with $A = 3$.

First, how far does the answer move when the input moves? Plot $y^*$ against $E(R_P)$ over
$\pm 2$ percentage points, and against $\sigma_P$ over $\pm 4$ percentage points, in two
panels. Then put a number on each: compute the elasticity of $y^*$ with respect to the risk
premium and with respect to volatility, either from the formula or numerically. The two
numbers are small integers.

Second, how much does the error cost? The loss in utility from holding $y$ instead of $y^*$
is a quadratic,

$$U(y^*) - U(y) = \tfrac{1}{2} A \sigma_P^2 (y - y^*)^2 ,$$

and a utility score is measured in return units, so the loss is readable in basis points of
certainty-equivalent return. Plot $U(y)$ for $A = 3$ and shade the range of $y$ that costs
less than 10 basis points. Report the width of that range in percentage points, and the cost
of two specific mistakes: holding $y = 1$ instead of $y^*$, and keeping $y^*$ unchanged
after a revision that raises $\sigma_P$ from 15.87 % to 17.87 %.

*Deliverable: the two-panel sensitivity figure with the two elasticities, the no-regret
width in percentage points, and the two costs in basis points.*

In [ ]:
A = 3
y_star_P = (ER_P - RF) / (A * SD_P**2)
fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.0, 4.0))
ers = np.linspace(ER_P - 0.02, ER_P + 0.02, 200)
sds = np.linspace(SD_P - 0.04, SD_P + 0.04, 200)
# TODO: y* as the expected return varies, then as the volatility varies
panels = [(ax1, ER_P, "Expected return of P"), (ax2, SD_P, "Volatility of P")]
for ax, x0, xlab in panels:
    ax.scatter([x0], [y_star_P], s=45, color=FHNW["navy"], zorder=3)
    ax.set_xlabel(xlab)
    ax.set_ylabel("Optimal share y*")
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))
fig3.tight_layout()
plt.show()
# TODO: the two elasticities, d log y* / d log x, at the published inputs

In [ ]:
# TODO: the curvature of the utility function in y, and the loss from holding y
# TODO: the half-width of the range that costs less than 10 basis points
fig4, ax = plt.subplots(figsize=(8.4, 4.4))
ax.plot(ys, utility_of_y(ys, ER_P, SD_P, A, RF), color=FHNW["navy"], lw=1.8)
ax.axvspan(y_star_P - half, y_star_P + half, color=FHNW["green"], alpha=0.15)
ax.axvline(y_star_P, color=FHNW["green"], lw=1.2)
ax.set_xlabel("Share y in AC World Equity")
ax.set_ylabel("Utility score of the complete portfolio")
plt.show()
# TODO: after the revision, the new y* and the cost of sticking with the old one

## Task 4 — Optional: what borrowing actually costs

Roger's $y^* = 1.26$ assumed he can borrow at the Swiss cash rate. He cannot. Suppose the
margin rate is $R_B = R_f + 1.5\ \%$, so the capital allocation line has a kink at $y = 1$:
lending below it, borrowing above it.

Compute the optimal $y$ under the kinked line for risk aversions from 1 to 8 and plot it
against $A$, with the unconstrained $y^*$ for comparison. There is an interval of $A$ over
which the answer is exactly $y = 1$. Find its endpoints and say in one sentence why an
interval appears at all.

*Deliverable: the figure, the two endpoints of the interval, and the sentence.*

In [ ]:
SPREAD = 0.015
As = np.linspace(1, 8, 200)
# TODO: y* using the lending rate, y* using the borrowing rate, then pick the
# TODO: one that is feasible on its own side of the kink; otherwise the corner y = 1
fig5, ax = plt.subplots(figsize=(8.4, 4.4))
ax.plot(As, y_lend, color="#999999", lw=1.4, ls="--",
        label="no borrowing spread")
ax.plot(As, y_kinked, color=FHNW["navy"], lw=2.0,
        label=f"borrowing at Rf + {SPREAD:.1%}")
ax.axhline(1.0, color=FHNW["red"], lw=1.0, ls=":")
ax.set_xlabel("Risk aversion A")
ax.set_ylabel("Optimal share y")
ax.legend()
plt.show()
# TODO: the two risk aversions at which the corner starts and ends

## Export

Bundle the figures and the tables, in case you want them for the transfer questions or your
own notes.

In [ ]:
# TODO: export the two tables together with the figures you want to keep

## Where this goes next

Two quizzes are open in Moodle until Sunday: the cumulative drill and the transfer
questions. Both are ungraded, and both are exactly the format the exams use.

Task 2 left one question standing. The ranking by $U^*$ said the whole menu should be held
in whichever single asset class has the highest Sharpe ratio, and no investor anywhere does
that. Lecture 04 supplies the missing ingredient: the assets on this menu move together
only partly, and a combination of them can have a higher Sharpe ratio than any of its
members.